In [1]:

import pandas as pd

# Load dataset
df = pd.read_csv("data/bank-full.csv", sep=";")

# Basic inspection
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())


Dataset loaded successfully!
Dataset shape: (45211, 17)

First 5 rows:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no



Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

Missing values:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [2]:

# Separate input features and target
X = df.drop(columns=["y", "duration"])
y = df["y"].map({"no": 0, "yes": 1})

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print((y.value_counts(normalize=True) * 100).round(2))


Features shape: (45211, 15)
Target shape: (45211,)

Target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Target percentage:
y
0    88.3
1    11.7
Name: proportion, dtype: float64


In [3]:

from sklearn.model_selection import train_test_split

# First: separate the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second: divide remaining data into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

print("\nPositive class proportion:")
print("Train:", round(y_train.mean() * 100, 2), "%")
print("Validation:", round(y_val.mean() * 100, 2), "%")
print("Test:", round(y_test.mean() * 100, 2), "%")


Training set: (27126, 15) (27126,)
Validation set: (9042, 15) (9042,)
Test set: (9043, 15) (9043,)

Positive class proportion:
Train: 11.7 %
Validation: 11.7 %
Test: 11.7 %


In [4]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

print("Numerical columns:", numerical_cols)
print("\nCategorical columns:", categorical_cols)
print("\nPreprocessor created successfully!")


Numerical columns: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Preprocessor created successfully!


C:\Users\Ujjawal\AppData\Local\Temp\ipykernel_30672\1526061504.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [5]:

# Training data par preprocessing rules learn + apply
X_train_processed = preprocessor.fit_transform(X_train)

# Validation aur test par wahi rules apply
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)


Train: (27126, 50)
Validation: (9042, 50)
Test: (9043, 50)


In [6]:

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

print("Model training completed!")


Model training completed!


In [12]:

# Validation data par predictions
y_val_pred = model.predict(X_val_processed)

# Har class ki predicted probability
y_val_prob = model.predict_proba(X_val_processed)[:, 1]

print("Predictions:", y_val_pred[:10])
print("Yes probabilities:", y_val_prob[:10])

Predictions: [1 0 0 0 0 0 0 1 0 0]
Yes probabilities: [0.795 0.09  0.075 0.23  0.005 0.045 0.265 0.59  0.06  0.345]


In [10]:

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, zero_division=0))
print("Recall:", recall_score(y_val, y_val_pred, zero_division=0))
print("F1 Score:", f1_score(y_val, y_val_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_val, y_val_prob))

print("\nClassification Report:\n")
print(classification_report(y_val, y_val_pred, zero_division=0))

print("Confusion Matrix:\n")
print(confusion_matrix(y_val, y_val_pred))


Accuracy: 0.8801150188011502
Balanced Accuracy: 0.6791661063147088
Precision: 0.4856828193832599
Recall: 0.4168241965973535
F1 Score: 0.4486266531027467
ROC-AUC: 0.7913783616382103

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.94      0.93      7984
           1       0.49      0.42      0.45      1058

    accuracy                           0.88      9042
   macro avg       0.70      0.68      0.69      9042
weighted avg       0.87      0.88      0.88      9042

Confusion Matrix:

[[7517  467]
 [ 617  441]]


In [13]:

import numpy as np
import pandas as pd

# Model ki predicted class aur actual class compare karo
correct = (y_val_pred == np.array(y_val))

# Har prediction ke liye model ka confidence
confidence = np.maximum(
    model.predict_proba(X_val_processed)[:, 0],
    model.predict_proba(X_val_processed)[:, 1]
)

results = pd.DataFrame({
    "Actual": np.array(y_val),
    "Predicted": y_val_pred,
    "Confidence": confidence,
    "Correct": correct
})

print(results.head(10))


   Actual  Predicted  Confidence  Correct
0       1          1       0.795     True
1       0          0       0.910     True
2       0          0       0.925     True
3       0          0       0.770     True
4       0          0       0.995     True
5       0          0       0.955     True
6       1          0       0.735    False
7       1          1       0.590     True
8       0          0       0.940     True
9       0          0       0.655     True


In [14]:

high_confidence_wrong = results[
    (results["Confidence"] >= 0.80) &
    (results["Correct"] == False)
]

print("Total validation predictions:", len(results))
print("High-confidence predictions:", (results["Confidence"] >= 0.80).sum())
print("High-confidence wrong predictions:", len(high_confidence_wrong))


Total validation predictions: 9042
High-confidence predictions: 6262
High-confidence wrong predictions: 361


In [15]:

# Confidence ke basis par groups banao
results["Confidence_Group"] = pd.cut(
    results["Confidence"],
    bins=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    include_lowest=True
)

# Har group ki accuracy aur predictions ki count
confidence_analysis = results.groupby(
    "Confidence_Group",
    observed=False
).agg(
    Total_Predictions=("Correct", "count"),
    Accuracy=("Correct", "mean")
)

# Accuracy ko percentage mein convert karo
confidence_analysis["Accuracy"] *= 100

print(confidence_analysis.round(2))


                  Total_Predictions  Accuracy
Confidence_Group                             
(0.499, 0.6]                    764     58.90
(0.6, 0.7]                      831     72.32
(0.7, 0.8]                     1279     85.38
(0.8, 0.9]                     2540     92.05
(0.9, 1.0]                     3628     95.84
